# 2. Generating Ground Truth Data

In [1]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

True

In [2]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [3]:
print(documents[0]['id'])
print(documents[0]['question'])

0e38656cfb
How do I submit homework?


Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [5]:
from openai import OpenAI

ollama_client = OpenAI(
    api_key='ollama',
    base_url='http://localhost:11434/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='granite4.1:3b'
    ):
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [17]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

['What is the process for turning in my assignments?', "Where can I find the specific folder for each semester's homework?", 'How do I access and use the submission forms provided by the course platform?', 'Are there any restrictions on when I can view my submitted answers?', 'Could you clarify where exactly I should publish my code related to the homework?']


Parallel processing

In [6]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

CACHE_DIR = Path('../../data/ground_truth')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    cache_file = CACHE_DIR / f"{doc['id']}.json"

    if cache_file.exists():
        return json.loads(cache_file.read_text())

    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = [
        {'question': q, 'course': doc['course'], 'document': doc['id']}
        for q in out.questions
    ]

    cache_file.write_text(json.dumps(results, ensure_ascii=False, indent=2))
    return results

Generate questions for all documents:

In [7]:
with ThreadPoolExecutor(max_workers=6) as pool:
    ground_truth = map_progress(pool, documents, process)

  0%|          | 0/1208 [00:00<?, ?it/s]

Flatten the nested lists into a single dataset:

In [8]:
import pandas as pd

ground_truth_flat = [item for sublist in ground_truth for item in sublist]
df_ground_truth = pd.DataFrame(ground_truth_flat)

print(len(df_ground_truth))

6020


In [9]:
# Save it for later use:
# df_ground_truth.to_csv(Path('../../data')/'ground-truth-data.csv', index=False)

# 3. Search Evaluation

Let's set up our search using RAGBase from module 01:

In [12]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

# Load documents
loader = FaqHttpLoader()
documents = loader.load()

# Index
index = MinsearchIndex(documents)

# LLM-client (Ollama, local)
llm_client = OllamaClient()

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:8b',
    instructions=instructions,
)

We'll use assistant.search to evaluate different boost configurations.

In [16]:
def search_fn(query, course):
    return assistant.search(
        query,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': course},
    )

Collecting relevance data

In [25]:
relevance_total = []

for q in tqdm(ground_truth_flat):
    doc_id = q['document']
    results = search_fn(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/6020 [00:00<?, ?it/s]

**Hit Rate**

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

$$\text{Hit Rate} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \mathbb{1}(q_i \in R(q_i))$$

Where $Q$ is the set of queries, $R(q_i)$ is the set of retrieved documents for query $q_i$, and $\mathbb{1}$ is the indicator function.

In [32]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

hit_rate_value = hit_rate(relevance_total)
print(f"Hit Rate (Recall@k): {hit_rate_value:.3f}")

Hit Rate (Recall@k): 0.770


**Mean Reciprocal Rank (MRR)**

In [33]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)


mrr_value = mrr(relevance_total)
print(f"MRR: {mrr_value:.3f}")

MRR: 0.632


Putting it together

In [36]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }



In [37]:
evaluate(
    ground_truth_flat,
    lambda q: search_fn(q['question'], q['course'])
)

  0%|          | 0/6020 [00:00<?, ?it/s]

{'hit_rate': 0.770265780730897, 'mrr': 0.63217331118494}

Try different boost values to see what works best:

In [39]:
def search_boost(query, course, boost_val):
    return assistant.search(
        query,
        boost_dict={'question': 1.0, "answer": boost_val, 'section': 0.5},
        filter_dict={'course': course},
    )
{"question": 3, "answer": 1, "section": 0.5}
for boost in [1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth_flat,
        lambda q: search_boost(q['question'], q['course'], boost)
    )
    print(f'boost={boost}: {result}')

  0%|          | 0/6020 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.838704318936877, 'mrr': 0.7145044296788488}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8906976744186047, 'mrr': 0.7798172757475076}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8813953488372093, 'mrr': 0.7715005537098552}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8654485049833887, 'mrr': 0.7551135105204867}


# 4. RAG Evaluation: Cosine Similarity

**Generating RAG answers**

In [10]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

documents = FaqHttpLoader().load()
index = MinsearchIndex(documents)
llm_client = OllamaClient(num_ctx=16384) # num_ctx=2048

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:3b',
    instructions=instructions,
)

Now run RAG on all ground truth questions and collect both the LLM answer and the original answer:

In [11]:
doc_idx = {d['id']: d for d in documents}

In [12]:
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def process_record(args):
    i, rec = args
    answer_llm = assistant.rag(
        rec['question'],
        filter_dict={'course': rec['course']}
    )
    doc_id = rec['document']
    return i, {
        'answer_llm': answer_llm,
        'answer_orig': doc_idx[doc_id]['answer'],
        'document': doc_id,
        'question': rec['question'],
        'course': rec['course'],
    }


ANSWERS_PATH = Path('../../data/answers.csv')

if ANSWERS_PATH.exists():
    df_answers = pd.read_csv(ANSWERS_PATH)
    answers = {i: row for i, row in enumerate(df_answers.to_dict('records'))}
    print(f"Loaded {len(answers)} answers from cache")
else:
    answers = {}
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(process_record, (i, rec)): i
                   for i, rec in enumerate(ground_truth_flat)
                   if i not in answers}

        for future in tqdm(as_completed(futures), total=len(futures)):
            i, result = future.result()
            answers[i] = result

    df_answers = pd.DataFrame(answers.values())
    ANSWERS_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_answers.to_csv(ANSWERS_PATH, index=False)
    print(f"Saved {len(answers)} answers to {ANSWERS_PATH}")


  0%|          | 0/6020 [00:00<?, ?it/s]

**Cosine similarity**

First, we turn both answers into vectors using an embedding model:

In [14]:
from sentence_transformers import SentenceTransformer

model_name = 'multi-qa-MiniLM-L6-cos-v1'
embedding_model = SentenceTransformer(model_name)

/home/ubuntu/repos/DTC/rag-app-with-llms/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12030). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Then for each answer pair, we encode both and compute the dot product:

In [15]:
import numpy as np

results = []

for i, rec in answers.items():
    v_llm = embedding_model.encode(rec['answer_llm'])
    v_orig = embedding_model.encode(rec['answer_orig'])
    score = v_llm.dot(v_orig)

    results.append({
        'answer_llm': rec['answer_llm'],
        'answer_orig': rec['answer_orig'],
        'cosine': score,
        'document': rec['document'],
        'question': rec['question'],
        'course': rec['course'],
    })

df_results = pd.DataFrame(results)

Let's check the average:

In [16]:
df_results['cosine'].describe()

count    6020.000000
mean        0.672680
std         0.188036
min        -0.159448
25%         0.564565
50%         0.710862
75%         0.813077
max         1.000000
Name: cosine, dtype: float64

A typical result for a working RAG system might be around 0.7-0.8. Lower values suggest the LLM is not using the context well, or the search is returning irrelevant documents.

**Comparing models**

In [21]:
import random

random.seed(42)
ground_truth_sample = random.sample(ground_truth_flat, 100)

models = ['granite4.1:3b', 'granite4.1:8b']

for model_name in models:
    assistant_model = RAGBase(
        index=index,
        llm_client=llm_client,
        instructions=instructions,
        llm_model=model_name,
    )

    answers_model = {}

    for i, rec in enumerate(tqdm(ground_truth_sample)):
        answer_llm = assistant_model.rag(
            rec['question'],
            filter_dict={'course': rec['course']}
        )
        doc_id = rec['document']
        original_doc = doc_idx[doc_id]
        answer_orig = original_doc['answer']

        answers_model[i] = {
            'answer_llm': answer_llm,
            'answer_orig': answer_orig,
        }
    # TODO clear ollama model from memory
    requests.post(
        'http://localhost:11434/api/chat',
        json={
            'model': 'granite4.1:3b',
            'keep_alive': 0,
            }
            )
    print('Model unloaded from memory')
    cosines = []
    for rec in answers_model.values():
        v_llm = embedding_model.encode(rec['answer_llm'])
        v_orig = embedding_model.encode(rec['answer_orig'])
        cosines.append(v_llm.dot(v_orig))

    print(f'{model_name}: mean cosine = {np.mean(cosines):.3f}')

  0%|          | 0/100 [00:00<?, ?it/s]

granite4.1:3b: mean cosine = 0.667


  0%|          | 0/100 [00:00<?, ?it/s]

granite4.1:8b: mean cosine = 0.682


# 5. LLM as a Judge

Q->A evaluation

In [23]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description='Step-by-step reasoning about the quality of the answer.'
    )
    score: Literal['good', 'bad'] = Field(
        description="Overall quality: 'good' if the answer is correct and complete, 'bad' otherwise."
    )

qa_judge_instructions = """
You are an expert evaluator. You will be given a question and an answer.

Evaluate the answer based on:
- Does it actually answer the question?
- Is it factually correct?
- Is it complete or does it miss important parts?

Be fair: a good answer doesn't need to be perfect. It needs to be
correct and helpful. Small formatting issues are not a problem.

Mark 'good' if the answer is correct and addresses the question.
Mark 'bad' only if the answer is wrong, incomplete, or off-topic.
""".strip()

qa_judge_prompt = """
Question:
{question}

Answer:
{answer}
""".strip()

In [24]:
def evaluate_qa(question, answer, model='granite4.1:3b'):
    prompt = qa_judge_prompt.format(question=question, answer=answer)

    messages = [
        {'role': 'system', 'content': qa_judge_instructions},
        {'role': 'user', 'content': prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=AnswerEvaluation,
        # max_tokens=1024,
    )

    return response.choices[0].message.parsed

result = evaluate_qa(
    question='How do I run Docker on Windows?',
    answer='To run Docker on Windows, install Docker Desktop for Windows.'
)
print(result.score, result.reasoning)

good The provided answer directly addresses the question by instructing to install Docker Desktop for Windows, which is the common and straightforward method to run Docker on a Windows operating system. It is factually correct as Docker Desktop includes all necessary components (Docker Engine, Kitematic, Compose) packaged together with ease of use interfaces tailored for Windows users. The response is complete enough for someone unfamiliar with Docker setup to proceed without needing additional technical details.


**A->Q->A' evaluation**

In [25]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [32]:
def evaluate_aqa(question, answer_orig, answer_llm, model='granite4.1:3b'):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    messages = [
        {'role': 'system', 'content': aqa_judge_instructions},
        {'role': 'user', 'content': prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=AnswerEvaluation,
        extra_body={'num_ctx': 8192},
    )

    return response.choices[0].message.parsed

Running the evaluation on 400 randomly selected results:

In [53]:
EVAL_PATH = Path('../../data/evaluations_aqa.csv')

if EVAL_PATH.exists():
    df_eval = pd.read_csv(EVAL_PATH)
    print(f'Loaded {len(df_eval)} evaluations from cache')
else:
    def evaluate_record(item):
        i, rec = item
        eval_result = evaluate_aqa(
            question=rec['question'],
            answer_orig=rec['answer_orig'],
            answer_llm=rec['answer_llm']
        )
        return {
            'question': rec['question'],
            'document': rec['document'],
            'score': eval_result.score,
            'reasoning': eval_result.reasoning,
        }

    # sample = random.sample(list(answers.items()), min(400, len(answers)))
    answers_sample = dict(random.sample(list(answers.items()), min(400, len(answers))))

    with ThreadPoolExecutor(max_workers=4) as pool:
        evaluations = list(tqdm(
            pool.map(evaluate_record, answers_sample.items()),
            total=len(answers_sample)
        ))

    df_eval = pd.DataFrame(evaluations)
    df_eval.to_csv(EVAL_PATH, index=False)
    print(f'Saved {len(df_eval)} evaluations to {EVAL_PATH}')

  0%|          | 0/400 [00:00<?, ?it/s]

Saved 400 evaluations to ../../data/evaluations_aqa.csv


Let's check the results:

In [54]:
good_count = (df_eval['score'] == 'good').sum()
total_count = len(df_eval)
print(f'Good: {good_count}/{total_count} = {good_count/total_count:.2%}')

Good: 355/400 = 88.75%


also look at the "bad" cases to understand what went wrong:

In [55]:
df_eval[df_eval['score'] == 'bad'][["question", "reasoning"]].head()

,question,reasoning
9,What are the necessary volume mappings require...,The original answer details specific Docker fl...
14,"In the context of this course, under which mod...",The original answer specifies the exact method...
15,Where can I find detailed documentation for th...,The original answer specifies that detailed do...
26,In which module of the MLOps Zoomcamp course i...,The original answer specifies a particular war...
28,Will Python 3.9 work for all components of the...,The original answer specifies that while Pytho...


**Combining metrics**

In [57]:
combined = []

for eval_rec, (i, rec) in zip(evaluations, answers_sample.items()):
    v_llm = embedding_model.encode(rec['answer_llm'])
    v_orig = embedding_model.encode(rec['answer_orig'])
    cosine = v_llm.dot(v_orig)

    combined.append({
        'question': rec['question'],
        'cosine': cosine,
        **eval_rec
    })

df_combined = pd.DataFrame(combined)

Look at the correlation between cosine similarity and judge scores:

In [58]:
df_combined.groupby('score')['cosine'].describe()

,count,mean,std,min,25%,50%,75%,max
score,,,,,,,,
bad,45.0,0.504407,0.223325,0.059465,0.367546,0.520475,0.656400,0.864096
good,355.0,0.699403,0.178760,-0.044699,0.595192,0.730449,0.835659,0.992298


If "good" answers have high cosine and "bad" answers have low cosine, both metrics agree. If they disagree, the judge might be catching things that cosine misses (or vice versa).

# 6. Collecting Agent Data

Setting up the agent

In [60]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

documents = FaqHttpLoader().load()
index = MinsearchIndex(documents)
llm_client = OllamaClient()

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:3b',
    instructions=instructions,
)

def search(query, course='data-engineering-zoomcamp'):
    return assistant.search(
        query,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': course},
    )

The tool schema and developer prompt define how the agent interacts with the search:

In [61]:
import json

search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [62]:
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [63]:
def make_call(tool_call):
    """The helper that executes a tool call and returns the result"""
    args = json.loads(tool_call.function.arguments)
    f_name = tool_call.function.name
    f = globals()[f_name]
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
    }


**Running the agent with logging**

modify the agent loop to collect the tool call trajectory:

In [64]:
def agent_with_logging(question, model='granite4.1:3b'):
    chat_messages = [
        {'role': 'system', 'content': developer_prompt},
        {'role': 'user', 'content': question}
    ]

    tools = []
    answer = None

    while True:
        response = ollama_client.chat.completions.create(
            model=model,
            messages=chat_messages,
            tools=[{'type': 'function', 'function': search_tool}],
        )

        message = response.choices[0].message
        chat_messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                tools.append({
                    'name': tool_call.function.name,
                    'args': tool_call.function.arguments
                })
                result = make_call(tool_call)
                chat_messages.append(result)
        else:
            answer = message.content
            break

    return {
        'question': question,
        'tools': tools,
        'answer': answer,
    }


The tools list records every function call the agent makes. This is the trajectory - the sequence of actions the agent took to answer the question.

**Collecting results**

In [65]:
from tqdm.auto import tqdm

doc_idx = {d['id']: d for d in documents}
agent_results = {}

for i, rec in enumerate(tqdm(ground_truth_flat[:50])):
    result = agent_with_logging(rec['question'])
    result['document'] = rec['document']
    result['course'] = rec['course']
    agent_results[i] = result

  0%|          | 0/50 [00:00<?, ?it/s]

Let's look at a result:

In [72]:
r = agent_results[5]
print('Question:', r['question'])
print('Tool calls:', len(r['tools']))
for t in r['tools']:
    print(f"  {t['name']}({t['args']})")
print('Answer:', r['answer'][:200], '...')

Question: Which framework replaces Flask for the deployment module in the 2025 edition?
Tool calls: 3
  search({"query":"Flask replaced deployment module 2025"})
  search({"query":"2025 data engineering course deployment module Flask replacement"})
  search({"query":"2025 Data Engineering course Flask deployment replacement"})
Answer: I wasn’t able to locate a specific FAQ entry that directly answers “Which framework replaces Flask for the deployment module in the 2025 edition?” using the available search tools. It seems the course ...


# 7. Trajectory Evaluation

Simple checks without LLM

In [73]:
def analyze_trajectory_simple(tools):
    num_calls = len(tools)

    queries = [t['args'] for t in tools]
    unique_queries = set(queries)
    num_duplicates = num_calls - len(unique_queries)

    return {
        'num_calls': num_calls,
        'num_duplicates': num_duplicates,
        'has_duplicates': num_duplicates > 0,
    }

for i, result in agent_results.items():
    analysis = analyze_trajectory_simple(result['tools'])
    result['num_calls'] = analysis['num_calls']
    result['has_duplicates'] = analysis['has_duplicates']

    print(f"[{i}] calls={analysis['num_calls']}, "
          f"duplicates={analysis['num_duplicates']}")

[0] calls=2, duplicates=1
[1] calls=1, duplicates=0
[2] calls=1, duplicates=0
[3] calls=2, duplicates=0
[4] calls=1, duplicates=0
[5] calls=3, duplicates=0
[6] calls=3, duplicates=0
[7] calls=1, duplicates=0
[8] calls=3, duplicates=0
[9] calls=2, duplicates=0
[10] calls=3, duplicates=0
[11] calls=0, duplicates=0
[12] calls=3, duplicates=0
[13] calls=0, duplicates=0
[14] calls=3, duplicates=0
[15] calls=0, duplicates=0
[16] calls=1, duplicates=0
[17] calls=1, duplicates=0
[18] calls=2, duplicates=0
[19] calls=1, duplicates=0
[20] calls=3, duplicates=0
[21] calls=1, duplicates=0
[22] calls=1, duplicates=0
[23] calls=3, duplicates=0
[24] calls=2, duplicates=0
[25] calls=1, duplicates=0
[26] calls=2, duplicates=0
[27] calls=1, duplicates=0
[28] calls=1, duplicates=0
[29] calls=2, duplicates=0
[30] calls=1, duplicates=0
[31] calls=2, duplicates=0
[32] calls=3, duplicates=0
[33] calls=1, duplicates=0
[34] calls=2, duplicates=0
[35] calls=1, duplicates=0
[36] calls=1, duplicates=0
[37] calls=

**LLM-based trajectory evaluation**

ask an LLM to judge the trajectory

define the output format:

In [74]:
from pydantic import BaseModel, Field
from typing import Literal

class TrajectoryResult(BaseModel):
    reasoning: str = Field(
        description='Step-by-step reasoning about the tool call trajectory.'
    )
    score: Literal['good', 'bad'] = Field(
        description="'good' if the trajectory was efficient, 'bad' if clearly wasteful."
    )
    suggestion: str = Field(
        description="How the agent could be more efficient, or 'none' if optimal."
    )

The judge instructions tell the LLM what to look for:

In [75]:
trajectory_instructions = """
You are an expert evaluator. You will be given:
1. A user question
2. The sequence of tool calls the agent made (the 'trajectory')
3. The agent's final answer

The agent has one tool: search(query) -- searches the FAQ.

Evaluate the trajectory:
- Were the search queries relevant to the question?
- Were there DUPLICATE tool calls? (same query repeated)
- Were there clearly irrelevant searches?
- More than 5 search calls is usually excessive for simple questions.

Mark 'good' if the trajectory was reasonably efficient.
Mark 'bad' only if there are clear inefficiencies: duplicate calls,
completely irrelevant queries, or excessive tool use.
""".strip()

trajectory_prompt = """
User Question:
{question}

Tool Call Trajectory:
{tools}

Agent's Final Answer:
{answer}
""".strip()

In [76]:
def evaluate_trajectory(question, tools, answer, model='granite4.1:3b'):
    """The trajectory evaluation function"""
    tools_str = '\n'.join(
        f"{i+1}. {t['name']}({t['args']})"
        for i, t in enumerate(tools)
    ) or '(no tool calls)'

    prompt = trajectory_prompt.format(
        question=question,
        tools=tools_str,
        answer=answer
    )

    messages = [
        {'role': 'system', 'content': trajectory_instructions},
        {'role': 'user', 'content': prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=TrajectoryResult,
        extra_body={'num_ctx': 16384},
    )

    return response.choices[0].message.parsed

Now evaluate all trajectories:

In [80]:
for i, result in tqdm(agent_results.items()):
    traj_eval = evaluate_trajectory(
        question=result['question'],
        tools=result['tools'],
        answer=result['answer']
    )

    result['trajectory'] = traj_eval.score
    result['trajectory_reasoning'] = traj_eval.reasoning
    result['trajectory_suggestion'] = traj_eval.suggestion

  0%|          | 0/50 [00:00<?, ?it/s]

Check the results:

In [83]:
import pandas as pd

df_agent = pd.DataFrame(agent_results.values())

print('Trajectory scores:')
print(df_agent['trajectory'].value_counts())
print()
print('Bad trajectories:')
for _, row in df_agent[df_agent['trajectory'] == 'bad'].iterrows():
    print(f"  Q: {row['question'][:100]}...")
    print(f"  Suggestion: {row['trajectory_suggestion']}")
    print()

Trajectory scores:
trajectory
good    48
bad      2
Name: count, dtype: int64

Bad trajectories:
  Q: ...
  Suggestion: Consider providing specific search queries or additional context to form a more complete answer.

  Q: Is there a specific platform I need to use to sign up for the machine-learning-zoomcamp, and if so, ...
  Suggestion: Perform a single relevant search to avoid duplication.



**Combining simple and LLM checks**

In [86]:
for i, result in agent_results.items():
    issues = []
    if result['has_duplicates']:
        issues.append('duplicates')
    if result['num_calls'] > 5:
        issues.append('excessive')
    if result['trajectory'] == 'bad':
        issues.append('llm_flagged')

    result['issues'] = issues

clean = sum(1 for r in agent_results.values() if not r['issues'])
total = len(agent_results)
print(f'Clean trajectories: {clean}/{total}')

Clean trajectories: 46/50


# 8. Instruction Following

**Answer correctness**

use LLM-as-a-judge approach to evaluate answer quality

In [91]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description='Step-by-step reasoning about the quality of the answer.'
    )
    score: Literal['good', 'bad'] = Field(
        description="'good' if correct, 'bad' otherwise."
    )

We compare the agent's answer with the original FAQ answer:

In [92]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [93]:
def evaluate_aqa(question, answer_orig, answer_llm, model='granite4.1:3b'):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    messages = [
        {'role': 'system', 'content': aqa_judge_instructions},
        {'role': 'user', 'content': prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=AnswerEvaluation,
        extra_body={'num_ctx': 16384},
    )

    return response.choices[0].message.parsed

Evaluate agent answers:

In [94]:
for i, result in tqdm(agent_results.items()):
    doc_id = result['document']
    original_doc = doc_idx[doc_id]
    answer_orig = original_doc['answer']

    eval_result = evaluate_aqa(
        question=result['question'],
        answer_orig=answer_orig,
        answer_llm=result['answer']
    )

    result['correctness'] = eval_result.score
    result['correctness_reasoning'] = eval_result.reasoning

**Instruction following**

Check if the agent follows its developer prompt rules:

In [97]:
class InstructionResult(BaseModel):
    reasoning: str = Field(
        description='Step-by-step reasoning about instruction following.'
    )
    score: Literal['good', 'bad'] = Field(
        description="'good' if instructions were followed, 'bad' if violated."
    )

instruction_instructions = """
You are an expert evaluator. You will be given:
1. The system instructions given to an agent
2. A user question
3. The agent's response

Check if the agent's response follows the instructions. Read the
instructions carefully and check each applicable rule.

Mark 'good' if the agent followed its instructions overall.
Mark 'bad' only if the agent clearly violated one or more rules.
""".strip()

instruction_prompt = """
=== AGENT INSTRUCTIONS ===
{instructions}

=== USER QUESTION ===
{question}

=== AGENT ANSWER ===
{answer}
""".strip()

In [99]:
def evaluate_instructions(question, answer, model='granite4.1:3b'):
    prompt = instruction_prompt.format(
        instructions=developer_prompt,
        question=question,
        answer=answer
    )

    messages = [
        {'role': 'system', 'content': instruction_instructions},
        {'role': 'user', 'content': prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=InstructionResult,
        extra_body={'num_ctx': 16384},
    )

    return response.choices[0].message.parsed

# Evaluate:
for i, result in tqdm(agent_results.items()):
    inst_eval = evaluate_instructions(
        question=result['question'],
        answer=result['answer']
    )

    result['instruction_following'] = inst_eval.score
    result['instruction_reasoning'] = inst_eval.reasoning

  0%|          | 0/50 [00:00<?, ?it/s]

**Putting it all together**

three evaluation dimensions for each agent run:

In [102]:
df_agent = pd.DataFrame(agent_results.values())

print('Correctness:', (df_agent['correctness'] == 'good').mean())
print('Trajectory:', (df_agent['trajectory'] == 'good').mean())
print('Instructions:', (df_agent['instruction_following'] == 'good').mean())

Correctness: 0.78
Trajectory: 0.96
Instructions: 0.98


These three metrics give you a comprehensive view of agent quality:

- Correctness: is the answer right?
- Trajectory: is the agent using tools well?
- Instructions: is the agent following its rules?

If correctness is low, the search or LLM might need improvement. If trajectory is bad, the agent is wasting calls - maybe the developer prompt needs better guidance. If instruction following is poor, the prompt rules might need to be clearer.

For production systems, you'd run these evaluations on a larger dataset and track the metrics over time as you iterate on the agent.